In [36]:
import csv

# 1️⃣ Charger le mapping
mapping = {}

with open("mapping.csv", newline="", encoding="utf-8") as map_file:
    reader = csv.reader(map_file)
    next(reader)  # retirer si pas d'en-tête
    for old, new in reader:
        mapping[old] = new


# 2️⃣ Lire le fichier comme TEXTE
with open(r"C:\Users\L14\Desktop\LIVRABLES_02_03_2026\ALL_CSV\ALL_TABLES_EXPORT bossoh.csv", "r", encoding="utf-8") as infile:
    content = infile.read()


# 3️⃣ Remplacer toutes les anciennes valeurs
for old, new in mapping.items():
    content = content.replace(old, new)


# 4️⃣ Écrire le nouveau fichier
with open(r"C:\Users\L14\Desktop\LIVRABLES_02_03_2026\ALL_CSV\ALL_TABLES_EXPORT bossoh_copie.csv", "w", encoding="utf-8") as outfile:
    outfile.write(content)

In [ ]:
import pandas as pd
from pathlib import Path

df_total = pd.read_csv(Path("C:\Users\L14\Downloads\Agent_CTB.xlsx"))

df = df_total.groupby()

In [37]:
import pandas as pd
from pathlib import Path
from unidecode import unidecode

folder = Path(r"C:\Users\L14\Desktop\CONTROLE_FINAL\Nouveau dossier")
MAIN_PATH = Path(r"C:\Users\L14\Desktop\LIVRABLES_02_03_2026")
pub_ouv_path = Path(fr"{MAIN_PATH}\PUB OUVERTE_total_fusionne.xlsx")
path_voisins_signes = Path(fr"{MAIN_PATH}\VOISINS_PRESENCE_total_fusionne.xlsx")

files = folder.glob('*.csv')

df_pub_ouv = pd.read_excel(pub_ouv_path, engine='openpyxl')
df_voisins_signes = pd.read_excel(path_voisins_signes, engine='openpyxl')

for file in files:
    df = pd.read_csv(file, sep=";", encoding='utf-8-sig')
    
    print(df.columns)
    if "A ajouter ou Corriger sur DIGIFOR" in df.columns:
        col = df["A ajouter ou Corriger sur DIGIFOR"].astype(str)

        df["indice"] = col.str.split(":").str[0].str.strip()
        df["nom_prenoms"] = col.str.split(":").str[1].str.strip()
        df["a_signer"] = col.str.contains(r"\(", regex=True, na=False).map({True: "OUI", False: "NON"})

    df = df.merge(
        df_pub_ouv[["codeAdvertisement","openDate"]],
        left_on='code',
        right_on='codeAdvertisement',
        how='inner'
    )

    df.to_excel(f"{folder}/{file.stem}_pub_ouv.xlsx")

df_voisins_ctb_signes = pd.read_excel(r"C:\Users\L14\Desktop\CONTROLE_FINAL\VOISINS CTB SIGNES_total_fusionne_pub_ouv.xlsx", engine='openpyxl')

df_voisins_signes.loc[:, "village"] = (df_voisins_signes["village"]
                                        .apply(lambda x: unidecode(x) if pd.notna(x) else x)
                                        .str.lower()
                                        .str.strip()
                                    )

df_voisins_signes.loc[:, "code_vois"] = (
        df_voisins_signes["village"].astype(str).str.strip()
        + "-"
        + df_voisins_signes["nameOfPerson"].astype(str).str.strip()
    )

df_voisins_signes = (
    df_voisins_signes
    .drop_duplicates("code_vois", keep="first")
)

df_voisins_ctb_signes.loc[:, "village"] = (df_voisins_ctb_signes["village"]
                                        .apply(lambda x: unidecode(x) if pd.notna(x) else x)
                                        .str.lower()
                                        .str.strip()
                                    )


df_voisins_ctb_signes.loc[:, "code_vois_old"] = (
    df_voisins_ctb_signes["village"].astype(str).str.strip()
    + "-"
    + df_voisins_ctb_signes["nom_prenoms"].astype(str).str.strip()
)

df_voisins_ctb_signes = df_voisins_ctb_signes.merge(
    df_voisins_signes[["code_vois", "numberCNI", "signatoryPhoto","representant"]],
    left_on="code_vois_old",
    right_on="code_vois",
    how="left",
    suffixes=("","_autre")
)


#df_voisins_ctb_signes.to_csv(r"C:\Users\L14\Desktop\CONTROLE_FINAL\VOISINS CTB SIGNES_total_fusionne_pub_ouv_signe.csv", sep=";", encoding="utf-8-sig")




Index(['code', 'code_parcelle', 'village', 'date', 'ancien_nom', 'nouveau_nom',
       'type_voisin', 'position', 'action', 'commentaire'],
      dtype='object')
Index(['code', 'code_parcelle', 'village', 'date', 'ancien_nom', 'nouveau_nom',
       'type_voisin', 'code_voisin', 'position', 'action', 'commentaire', '﻿',
       'code.1', 'position_ctb', 'code_par', 'nom_a_ajouter', 'name',
       'cni_voisin', 'signature_voisin', 'nameOfPersonOriginal',
       'position_dig', 'description', 'nameOfNeighborOriginal', 'num_voisin',
       'description_autre'],
      dtype='object')
